# 💧 AquaSentinel AI - watsonx.ai Anomaly Modeling & Granite Time Series
### Sociothon'26 | PRISM | 1M1B Hackathon
**Team**: Social Spark (COEP, Pune)
**Theme**: Smart Cities & Water Management (SDG 6 · SDG 11 · SDG 12)

This notebook demonstrates the end-to-end pipeline referenced in the project architecture:
1. **Pipeline Sensor Data Ingestion** (Pressure & Flow readings)
2. **Data Processing & Diurnal Baseline Creation** (Expected daily cyclic demand)
3. **AI-based Anomaly Detection** (Granite TSPulse residual scoring & Isolation Forest)
4. **Multi-Sensor Leak Assessment** (Distinguishing localized pipe rupture vs supply drops)
5. **Explainable Risk Prioritization** (Multilingual operator narratives)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest

print("Libraries loaded successfully.")

## 1. Load Telemetry Data (BattLeDIM / SCADA Format)

In [2]:
df_leak = pd.read_csv('../data/sample_leak_incident.csv')
df_leak['timestamp'] = pd.to_datetime(df_leak['timestamp'])
df_leak.head(10)

## 2. Diurnal Baseline Calculation
Expected pressure $\mu_P(t)$ and flow $\mu_Q(t)$ are modeled using municipal diurnal demand patterns.

In [3]:
nominal_pressure = 5.20
nominal_flow = 280.0

df_leak['expected_pressure'] = nominal_pressure
df_leak['expected_flow'] = nominal_flow

# Deviations
df_leak['pressure_dev_pct'] = ((df_leak['pressure'] - df_leak['expected_pressure']) / df_leak['expected_pressure']) * 100.0
df_leak['flow_dev_pct'] = ((df_leak['flow'] - df_leak['expected_flow']) / df_leak['expected_flow']) * 100.0

df_leak[['timestamp', 'pressure', 'pressure_dev_pct', 'flow', 'flow_dev_pct', 'event_annotation']]

## 3. IBM Granite Time Series (TSPulse) Residual Anomaly Scoring
Granite TSPulse calculates non-linear residual discrepancy between the observed trajectory and predicted nominal bounds.

In [4]:
def compute_granite_tspulse_score(p_obs, p_exp, q_obs, q_exp):
    res_p = abs(p_obs - p_exp) / p_exp
    res_q = abs(q_obs - q_exp) / q_exp
    # Leak signature: pressure drops while flow increases
    if (p_obs < p_exp) and (q_obs > q_exp):
        score = min(1.0, (res_p * 0.6 + res_q * 0.4) * 2.5)
    else:
        score = min(1.0, (res_p * 0.5 + res_q * 0.5))
    return round(score, 3)

df_leak['tspulse_score'] = df_leak.apply(
    lambda r: compute_granite_tspulse_score(r['pressure'], r['expected_pressure'], r['flow'], r['expected_flow']),
    axis=1
)
df_leak['anomaly_flag'] = df_leak['tspulse_score'] > 0.40
df_leak[['timestamp', 'pressure', 'flow', 'tspulse_score', 'anomaly_flag', 'event_annotation']]

## 4. Multi-Sensor Leak vs Supply Interruption Classification
Distinguishing localized pipe burst from broader pump or reservoir failure.

In [5]:
def classify_event(p_dev, q_dev):
    if p_dev <= -15.0 and q_dev >= 10.0:
        return 'LOCALISED_LEAK (High Risk)'
    elif p_dev <= -12.0 and q_dev <= -10.0:
        return 'BROADER_SUPPLY_ISSUE'
    else:
        return 'NORMAL_DEMAND_FLUCTUATION'

df_leak['classification'] = df_leak.apply(
    lambda r: classify_event(r['pressure_dev_pct'], r['flow_dev_pct']),
    axis=1
)
df_leak[['timestamp', 'pressure_dev_pct', 'flow_dev_pct', 'classification']]

## 5. Visualizing the Leak Onset and Intervention (Matplotlib)

In [6]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

ax1.plot(df_leak['timestamp'], df_leak['pressure'], marker='o', color='#38bdf8', label='Observed Pressure (bar)')
ax1.axhline(5.20, color='#10b981', linestyle='--', label='Baseline Expected')
ax1.set_ylabel('Pressure (bar)')
ax1.set_title('AquaSentinel AI: Section B Pipe Burst Detection & Recovery')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(df_leak['timestamp'], df_leak['flow'], marker='s', color='#f59e0b', label='Observed Flow (m³/h)')
ax2.axhline(280.0, color='#10b981', linestyle='--', label='Baseline Expected')
ax2.set_ylabel('Flow Rate (m³/h)')
ax2.set_xlabel('Timestamp')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()